In [1]:
import pandas as pd

#from https://medium.com/parsing-bulk-whois-data/parsing-ripe-bulk-whois-data-8495d5fb5fe9
#By Thomas Gorman
ripe = pd.read_table("ripe.db", encoding="ISO-8859-1")
ripe.columns = ["data"]
ripe = ripe[ripe.data.str.contains("remarks") == False]
ripe = ripe[ripe.data.str.contains("%") == False]
inetnum_loc = ripe.data.str.contains("inetnum").idxmax()
ripe = ripe[inetnum_loc:]
ripe = ripe.reset_index(drop=True)


In [2]:
print(ripe.head())

                                   data
0      nserver:        ns2.lewtelnet.de
1         mnt-by:         LEWTELNET-MNT
2  created:        1970-01-01T00:00:00Z
3  last-modified:  2004-04-01T13:53:04Z
4                  source:         RIPE


In [3]:
s = pd.Series(ripe["data"])
ripe2 = ripe["data"].str.split(pat = ":", n = 1, expand=True)

In [ ]:
ripe2 = ripe2.set_index([0, (ripe[0] == "inetnum").cumsum().rename("row")])
ripe2 = ripe2.set_index(ripe.groupby([0, "row"]).cumcount(), append = True)
ripe2 = ripe2.reset_index("row")
ripe2.index = ripe2.index.map("{0[0]}_{0[1]}".format)
ripe2 = ripe2.set_index(["row"], append = True)[1].unstack(0)
ripe2 = ripe2.rename(columns = lambda x: x.split("_0")[0]).reset_index()
print(ripe2.head())

KeyError: 0